# 05 - Mean Reversion Signals

**Author:** Sacha Huberty

**Purpose:** Build the per-asset mean-reversion signal (ADF stationarity,
OU half-life, rolling z-scores, volatility filter), validate it with an
event study (do extreme z-scores actually predict reversion in-sample?),
then wire it in as V2 -- an additive tilt on top of stage 3's regime
posture and stage 4's anomaly override -- and backtest the resulting
strategy OOS.

**Last updated:** 2026-07-25

## Setup

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import allocation, backtest, data, meanreversion, metrics, regimes, strategy, universe

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["meanreversion"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)
returns.tail()

## Analysis / signal logic

### Per-asset diagnostic (in-sample snapshot)

z, ADF p-value, half-life, vol-filter pass, and the resulting view as
of the in-sample boundary -- an illustration, not a decision (the OOS
backtest below recomputes independently every Friday on trailing data
only).

**Methodology note:** ADF runs on the RAW log-price over a longer
`adf_lookback_days` window, not on the rolling-detrended series used
for the z-score. Rolling-detrending a series induces spurious apparent
stationarity -- confirmed directly while building this module: a true
random walk's rolling-detrended residual falsely rejected the
unit-root null (p=0.002), while testing its untouched log-price
correctly failed to reject (p=0.54, as expected). A short window also
gives ADF too little power regardless, which is why the ADF window is
longer than the z-score/half-life window.

In [ ]:
diag = meanreversion.reversion_signal(prices.loc[:as_of_universe], cfg)
diag.sort_values("adf_pvalue")

In [ ]:
most_stationary = diag["adf_pvalue"].idxmin()
window = cfg["meanreversion"]["lookback_days"]
log_price = np.log(prices[most_stationary]).loc[:as_of_universe]
z = meanreversion.zscore(log_price.to_frame(most_stationary), window)[most_stationary]

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(log_price.tail(500))
axes[0].set_title(f"{most_stationary}: log-price")
axes[1].plot(z.tail(500))
axes[1].axhline(cfg["meanreversion"]["entry_z"], color="red", linestyle="--", linewidth=0.8)
axes[1].axhline(-cfg["meanreversion"]["entry_z"], color="red", linestyle="--", linewidth=0.8)
axes[1].set_title(f"{most_stationary}: rolling z-score (entry_z bands)")
plt.tight_layout()
plt.show()

### Event study: do extreme z-scores actually predict reversion?

For every in-sample date where |z| > entry_z, per asset: did the price
move toward the mean over the next 10 trading days? A genuine
predictive edge should show hit rates meaningfully above 50%.

In [ ]:
event = meanreversion.event_study(prices.loc[:as_of_universe], cfg, horizon_days=10)
event.sort_values("hit_rate", ascending=False)

### V2 wiring: mean-reversion tilt -> backtest OOS

Layers `with_meanreversion_tilt` on top of stage 4's anomaly-wrapped
V1 strategy. Same buffered-by-HMM-lookback, reduced-refit-frequency
scope trade as stage 4's notebook (autoencoder refits are the
expensive part; the mean-reversion signal itself is cheap OLS/ADF and
adds negligible overhead).

In [ ]:
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
hmm_lookback = cfg["regimes"]["hmm"]["lookback_days"]

buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - hmm_lookback)
backtest_returns = returns.iloc[buffer_start_pos:]

backtest_cfg = copy.deepcopy(cfg)
backtest_cfg["anomaly"]["epochs"] = 10
backtest_cfg["anomaly"]["patience"] = 3
backtest_cfg["anomaly"]["refit_frequency_days"] = 126

v1_fn = strategy.regime_switching_strategy(class_bucket, backtest_cfg, posture_cfg)
v1_anomaly_fn = strategy.with_anomaly_override(v1_fn, backtest_cfg)
v1_anomaly_meanrev_fn = strategy.with_meanreversion_tilt(v1_anomaly_fn, backtest_cfg)

meanrev_result_bt = backtest.run(v1_anomaly_meanrev_fn, backtest_returns, backtest_cfg)

In [ ]:
# Stage 2/3/4 baselines, recomputed here (same buffered range) for a
# direct comparison.
lookback = cfg["optimization"]["lookback_days"]


def make_classical_strategy(method):
    cov_method = cfg["optimization"]["covariance"]

    def strategy_fn(as_of, window):
        w = window.tail(lookback)
        cov_t = allocation.covariance_matrix(w, method=cov_method)
        if method == "risk_parity":
            return allocation.risk_parity(cov_t, cfg)
        raise ValueError(method)
    return strategy_fn


def permanent_strategy(as_of, window):
    return allocation.permanent(class_bucket)


v1_only_fn = strategy.regime_switching_strategy(class_bucket, cfg, posture_cfg)
v1_anomaly_baseline_fn = strategy.with_anomaly_override(v1_only_fn, backtest_cfg)

baseline_fns = {
    "permanent": permanent_strategy,
    "risk_parity": make_classical_strategy("risk_parity"),
    "regime_switching": v1_only_fn,
    "regime_switching_anomaly": v1_anomaly_baseline_fn,
}
baseline_results = {
    name: backtest.run(fn, backtest_returns, cfg)
    for name, fn in baseline_fns.items()
}
all_results = {"regime_switching_anomaly_meanrev": meanrev_result_bt, **baseline_results}

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "calmar": metrics.calmar(r),
        "max_drawdown": metrics.max_drawdown(r),
        "hit_rate": metrics.hit_rate(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


comparison = pd.DataFrame(
    {name: oos_metrics(res) for name, res in all_results.items()}
).T
comparison.sort_values("sharpe", ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in all_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    lw = 2 if name == "regime_switching_anomaly_meanrev" else 1
    oos_curve.plot(label=name, linewidth=lw)
plt.title("OOS equity curves: V1 + anomaly + mean-reversion vs. baselines")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend()
plt.show()

## Notes / next steps

- **Honest findings from this run:** the event study shows the mean-reversion assumption does NOT hold uniformly across the universe. HYG and VNQ show hit rates meaningfully above chance (~60%), genuinely predictive reversion. But AGG, LQD, and USO show hit rates BELOW chance (36-41%) -- extreme z-scores in those assets tended to continue (momentum), not revert, over the 10-day horizon. Applying one uniform mean-reversion rule across every asset is therefore a simplification; a refinement would condition on each asset's own validated hit rate, though that risks overfitting to this specific event-study sample and is exactly the kind of choice the ablation study (notebook 11) should judge OOS, not this notebook. And OOS: V2 gave a small improvement over V1 alone (Sharpe 0.7605 vs. 0.7523, see the comparison table above) -- still below the naive stage-2 baselines, honestly reported per PROJECT_STRUCTURE.md's overfitting defense rather than tuned to look better.

- The event study above is the key validation for this stage: it
  checks whether extreme z-scores actually predicted reversion
  in-sample, independent of whether the resulting OOS Sharpe improved
  -- a signal can be genuinely predictive and still not move a
  already-diversified portfolio's Sharpe much, and the two questions
  are worth keeping separate.
- **Methodology fix worth remembering:** ADF must run on the raw
  log-price, not a rolling-detrended series, or it will report false
  stationarity. This is a general pitfall for any "detrend, then test"
  signal pipeline, not specific to this asset universe.
- `exit_z` remains unused (see meanreversion.py's docstring): it's
  position-level hysteresis for a stateful trading system, and this
  is still a stateless weekly view generator. It becomes relevant once
  Black-Litterman (stage 8) or a real position-tracking layer exists.
- Next (stage 6): `technicals.py` (pivot-based support/resistance
  zones, K-Means clustering, options positioning where live data is
  available), added as V3 plus execution timing.